## LangChain Assignment: Build an Intelligent Travel Assistant AI


#### Enter Openai and Tavily and WebAPI Setup




In [ ]:
from getpass import getpass

OPENAI_KEY = getpass('Enter Open AI API Key: ')

In [ ]:
TAVILY_API_KEY = getpass('Enter Tavily Search API Key: ')

In [ ]:
WEATHER_API_KEY = getpass('Enter WeatherAPI API Key: ')

### Setup Environment Variables

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = OPENAI_KEY
os.environ['TAVILY_API_KEY'] = TAVILY_API_KEY

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool = TavilySearchResults(max_results=5,
                                  search_depth='advanced',
                                  include_answer=True,
                                  include_raw_content=False)

In [ ]:
from tqdm import tqdm
from langchain_core.tools import tool
import requests

@tool
def search_tourist_attractions(query: str) -> list:
    """
    Searches the web for tourist spots related to a given city and extracts
    useful information from the search results.
    """
    print('Calling search_tourist_attractions search tool')
    results = tavily_tool.invoke(f"Top tourist attractions in {query}")
    docs = ""
    for result in tqdm(results):
        try:
            docs += result['content']
        except Exception as e:
            print(f"Error extracting from url: {result['url']} - {e}")
    return docs

@tool
def search_travel_info(query: str) -> list:
    """
    Searches the web for travel information about a given city and extracts
    useful details for planning a trip.
    """
    print('Calling search_travel_info search tool')
    results = tavily_tool.invoke(
        f"Travel information about {query}: best time to visit, local customs, transportation, recommended activities"
    )
    docs = ""
    for result in tqdm(results):
        try:
            docs += result['content']
        except Exception as e:
            print(f"Error extracting from url: {result['url']} - {e}")
    return docs

@tool
def get_weather_forecast(query: str) -> dict | str:
    """
    Fetches the current weather for a given location using WeatherAPI.

    Args:
        query (str): The city, state, or tourist spot name to fetch weather for.

    Returns:
        dict: Weather data including temperature, condition, humidity, etc.,
              if the location is found.
        str: "Weather Data Not Found" if the location is invalid or not found.
    """
    print('Calling weather tool...')
    print(f"Location: {query}")

    base_url = "http://api.weatherapi.com/v1/current.json"
    complete_url = f"{base_url}?key={WEATHER_API_KEY}&q={query}"

    try:
        response = requests.get(complete_url, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("location"):
            return data
        else:
            return "Weather Data Not Found"

    except requests.exceptions.RequestException as e:
        return f"Error fetching weather data: {e}"

### Test Tool Calling with LLM

In [ ]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [search_tourist_attractions, search_travel_info,get_weather_forecast]
# bind_tools → Just says “Hey LLM, these tools exist, you can call them.”
chatgpt_with_tools = chatgpt.bind_tools(tools)

prompt = "kashmir"
response = chatgpt_with_tools.invoke(prompt)
response.tool_calls

[{'name': 'search_tourist_attractions',
  'args': {'query': 'Kashmir tourist attractions'},
  'id': 'call_o48EJMLxa76oyGUoB2LA2XM2',
  'type': 'tool_call'},
 {'name': 'search_travel_info',
  'args': {'query': 'Kashmir travel information'},
  'id': 'call_dlcP8OCtWb7W4PvLmm08bICW',
  'type': 'tool_call'},
 {'name': 'get_weather_forecast',
  'args': {'query': 'Kashmir'},
  'id': 'call_aWDsZZRqHwNWaWxFgxjkq9HP',
  'type': 'tool_call'}]

## Build and Test AI Agent

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYS_PROMPT = """Act as a helpful tourist guide and weather assistant.
                You run in a loop of Thought, Action, PAUSE, Observation.
                At the end of the loop, you output an Answer with tourist recommendations and weather forecast.
                Use Thought to describe your approach to finding tourist spots and weather for the requested city.
                Use Action to run one of the actions available to you - then return PAUSE.
                Observation will be the result of running those actions.
                Repeat till you have gathered sufficient information for both tourist spots and weather forecast.

                Use the following workflow format:
                  Question: the city name provided by user for tourist recommendations and weather
                  Thought: analyze what information I need (top tourist spots + 3-day weather forecast  + additional travel info such as best time to visit, customs, and transport)
                  Action: the action to take which can be any of the following:
                            - search for top tourist attractions and landmarks in the city
                            - get current weather and 3-day forecast for the city
                            - gather additional city information if needed (best time to visit, local tips)
                            - use my trained knowledge if I have reliable information about the city
                  Action Input: the specific query or city name for the action
                  Observation: the result of the action (tourist spots found, weather data, etc.)
                  ... (this Thought/Action/Action Input/Observation can repeat N times)
                  Thought: I now have sufficient information about tourist spots and weather
                  Final Answer: comprehensive response including:
                                - Top 5-10 tourist attractions with brief descriptions
                                - 3-day weather forecast with temperatures and conditions
                                - Best time to visit, local customs, and transportation tips
                                - Weather-based travel recommendations (what to pack, best activities for the weather)

                Tools at your disposal to perform tasks:
                  - get_weather_forecast: get current weather and 3-day forecast for any city
                  - search_tourist_attractions: find top tourist spots, landmarks, and attractions in a city
                  - search_travel_info: get additional travel information like best time to visit, local customs, transportation

                Always provide:
                1. At least 5 top tourist attractions with descriptions
                2. 3-day weather forecast with daily highs/lows and conditions
                3. Weather-based travel recommendations (what to pack, best activities for the weather)
                4. Any relevant seasonal or local travel tips
             """

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", SYS_PROMPT),
        MessagesPlaceholder(variable_name="history", optional=True),
        ("human", "{query}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

prompt_template.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Act as a helpful tourist guide and weather assistant.\n                You run in a loop of Thought, Action, PAUSE, Observation.\n                At the end of the loop, you output an Answer with tourist recommendations and weather forecast.\n                Use Thought to describe your approach to finding tourist spots and weather for the requested city.\n                Use Action to run one of the actions available to you - then return PAUSE.\n                Observation will be the result of running those actions.\n                Repeat till you have gathered sufficient information for both tourist spots and weather forecast.\n\n                Use the following workflow format:\n                  Question: the city name provided by user for tourist recommendations and weather\n                  Thought: analyze what information I need (top tourist spots + 3-day w

In [ ]:
from langchain.agents import create_tool_calling_agent

chatgpt = ChatOpenAI(model="gpt-4o", temperature=0)
tools = [search_tourist_attractions, search_travel_info,get_weather_forecast]
# create_tool_calling_agent → Builds a structured agent that can plan, reason, and call multiple tools step-by-step using a defined prompt loop.
# But this agent by itself is just logic — it doesn’t actually run the loop end-to-end.
# Analogy - Agent (from create_tool_calling_agent) = The pilot who knows how to fly.
agent = create_tool_calling_agent(chatgpt, tools, prompt_template)

In [ ]:
from langchain.agents import AgentExecutor
# AgentExecutor = The airplane cockpit that gives the pilot the instruments, rules (max_iterations), and executes the flight safely.
agent_executor = AgentExecutor(agent=agent,
                               tools=tools,
                               early_stopping_method='force',
                               max_iterations=10)

In [ ]:
query = """Goa"""
response = agent_executor.invoke({"query": query})


Calling search_tourist_attractions search tool


100%|██████████| 5/5 [00:00<00:00, 43329.59it/s]


Calling weather tool...
Location: Goa
Calling search_travel_info search tool


100%|██████████| 5/5 [00:00<00:00, 25668.94it/s]


In [ ]:
from IPython.display import display, Markdown

display(Markdown(response['output']))

Final Answer:

**Top Tourist Attractions in Goa:**

1. **Basilica of Bom Jesus**: A UNESCO World Heritage Site, this iconic Catholic church in Old Goa is known for its stunning architecture and the preserved body of St. Francis Xavier. It's a must-see for its historical and architectural significance.

2. **Fort Aguada**: A well-preserved 17th-century Portuguese fort standing on Sinquerim Beach, offering panoramic views of the Arabian Sea.

3. **Anjuna Beach**: Famous for its vibrant nightlife, flea markets, and beach parties, Anjuna Beach is a hotspot for tourists looking to experience Goa's lively beach culture.

4. **Dudhsagar Waterfalls**: One of India's tallest waterfalls, located on the Mandovi River, offering breathtaking views and a popular spot for trekking.

5. **Chapora Fort**: Known for its scenic views and historical significance, this fort is a popular spot for photography and exploring Goa's colonial past.

6. **Palolem Beach**: A picturesque beach in South Goa, known for its serene environment and beautiful crescent shape, ideal for relaxation and water activities.

7. **Spice Plantations**: Explore the aromatic spice plantations in Ponda, where you can learn about the cultivation of spices and enjoy a traditional Goan meal.

**3-Day Weather Forecast for Goa:**

- **Day 1**: Patchy rain nearby with a temperature of 25.6°C (78°F). Humidity is high at 86%, with winds at 15.1 kph (9.4 mph).
- **Day 2**: Expect similar conditions with occasional rain showers. Temperatures will range from 24°C to 29°C.
- **Day 3**: Partly cloudy with a chance of rain. Temperatures will be between 23°C and 28°C.

**Best Time to Visit:**

- The ideal time to visit Goa is from November to February when the weather is pleasant, making it perfect for beach activities, sightseeing, and enjoying the vibrant nightlife. This period also hosts numerous festivals and events, adding cultural charm to your visit.

**Local Customs and Transportation Tips:**

- **Customs**: Goa is known for its laid-back lifestyle and vibrant culture. Respect local traditions, especially when visiting religious sites. Dress modestly when entering churches and temples.
- **Transportation**: Renting a scooter or bike is a popular way to explore Goa. Taxis and auto-rickshaws are also available, but it's advisable to agree on fares beforehand.

**Weather-Based Travel Recommendations:**

- **Packing**: Bring light, breathable clothing, sunscreen, and a raincoat or umbrella for unexpected showers. Comfortable footwear is recommended for exploring beaches and forts.
- **Activities**: Enjoy water sports like parasailing and jet-skiing during clear weather. Visit indoor attractions like museums and spice plantations on rainy days.

Goa offers a unique blend of natural beauty, historical sites, and vibrant culture, making it a must-visit destination for travelers seeking both relaxation and adventure.

## Few Sample Questions



In [ ]:
samples = [
    "Plan a trip to Kashmir",
    "What can I see and do in Jaipur?",
    "Travel guide for Tokyo in March",
    "Best time to visit Dal Lake in Kashmir",
    "Tell me about Eiffel Tower Paris trip",
    "I’m visiting Goa next month, what should I know?",
    "What are the top things to do in New York and how’s the weather now?",
    "Give me a tourist guide for Singapore including attractions, travel tips, and weather",
    "When is the best time to visit Manali and what should I pack?",
    "Cultural tips and tourist places in Kyoto"
]
for q in samples:
    response = agent_executor.invoke({"query": q})
    display(Markdown(response['output']))
    print('='*150)

Calling search_tourist_attractions search tool


100%|██████████| 5/5 [00:00<00:00, 36220.24it/s]


Calling weather tool...
Location: Kashmir
Calling search_travel_info search tool


100%|██████████| 5/5 [00:00<00:00, 20184.33it/s]


Final Answer:

**Top Tourist Attractions in Kashmir:**
1. **Srinagar**: Known for its beautiful gardens, lakes, and houseboats. Key attractions include Dal Lake, Shikara rides, Mughal Gardens (Nishat and Shalimar), and the Shankaracharya Temple.
2. **Gulmarg**: Famous for skiing and the Gulmarg Gondola, one of the highest cable cars in the world. It's a winter sports hub.
3. **Pahalgam**: Known for its scenic beauty and as a base for the Amarnath Yatra. Enjoy activities like trekking and fishing.
4. **Sonmarg**: Offers stunning views of the Himalayas and is a gateway to the Thajiwas Glacier.
5. **Doodhpathri**: An unexplored gem with lush meadows and beautiful landscapes.

**3-Day Weather Forecast for Kashmir:**
- **Day 1**: Clear skies with a high of 30.7°C (87.3°F) and a low of 20°C (68°F). Ideal for outdoor activities.
- **Day 2**: Partly cloudy with temperatures ranging from 28°C (82°F) to 18°C (64°F). A good day for sightseeing.
- **Day 3**: Sunny with a high of 29°C (84°F) and a low of 19°C (66°F). Perfect for exploring the gardens and lakes.

**Best Time to Visit:**
- The best time to visit Kashmir is from March to August, during spring and summer, when the valley is in full bloom and the weather is pleasant. This period is ideal for sightseeing, family vacations, and outdoor activities.

**Local Customs and Transportation Tips:**
- Respect local customs and traditions. Learning a few phrases in the local language can be helpful.
- Book transportation and accommodations in advance, especially during peak tourist seasons.
- Consider staying in a houseboat on Dal Lake for a unique experience.

**Weather-Based Travel Recommendations:**
- Pack light clothing for the day and warmer layers for the evening.
- Carry sunscreen and sunglasses for protection against the sun.
- Engage in activities like Shikara rides, trekking, and visiting gardens, which are best enjoyed in clear weather.

Kashmir offers a blend of natural beauty, adventure, and cultural experiences, making it a must-visit destination. Enjoy your trip!

...

100%|██████████| 5/5 [00:00<00:00, 50051.36it/s]


Calling search_travel_info search tool


100%|██████████| 5/5 [00:00<00:00, 62977.54it/s]


Calling weather tool...
Location: Kyoto


### Tourist Attractions in Kyoto

1. **Kinkaku-ji (Golden Pavilion)**: This iconic Zen Buddhist temple is covered in gold leaf and surrounded by beautiful gardens. It's one of the most photographed sites in Kyoto.

2. **Fushimi Inari Taisha**: Famous for its thousands of red torii gates, this shrine is a must-visit. It's open 24 hours, so visiting early in the morning or late at night can help avoid crowds.

3. **Kiyomizu-dera**: A historic temple with a stunning wooden stage that offers panoramic views of Kyoto. It's especially beautiful during cherry blossom and autumn leaf seasons.

4. **Arashiyama Bamboo Grove**: Walk through towering bamboo stalks in this serene and picturesque grove. It's best visited early in the morning to avoid the crowds.

5. **Nijo Castle**: A UNESCO World Heritage Site, this castle is known for its beautiful gardens and the "nightingale floors" that chirp when walked upon.

6. **Gion District**: Known for its traditional wooden machiya houses, this area is famous for geisha sightings and traditional tea houses.

7. **Philosopher’s Path**: A scenic walkway lined with cherry trees, perfect for a leisurely stroll.

8. **Nishiki Market**: A bustling marketplace offering a variety of local foods and goods, ideal for experiencing Kyoto's culinary delights.

### Weather Forecast for Kyoto

- **Current Weather**: Partly cloudy with a temperature of 30.1°C (86.2°F). The humidity is high at 79%, making it feel warmer.

- **3-Day Forecast**:
  - **Day 1**: Expect partly cloudy skies with temperatures ranging from 28°C to 34°C (82°F to 93°F).
  - **Day 2**: Mostly sunny with a high of 33°C (91°F) and a low of 27°C (81°F).
  - **Day 3**: Scattered showers possible, with temperatures between 26°C and 32°C (79°F to 90°F).

### Travel Tips and Recommendations

- **Best Time to Visit**: Spring (March to May) and autumn (October to November) are ideal for visiting Kyoto due to the pleasant weather and beautiful seasonal changes. However, these are also peak tourist seasons, so plan accordingly.

- **Local Customs**: Kyoto is a cash-friendly city, and many places may not accept credit cards. Tipping is not customary in Japan.

- **Transportation**: Kyoto has an efficient public transportation system, including buses and trains. Renting a bicycle is also a popular way to explore the city.

- **Weather-Based Recommendations**: Given the warm and humid weather, pack light, breathable clothing, and stay hydrated. Early morning or late evening visits to popular sites can help avoid the heat and crowds. Consider carrying an umbrella for potential rain showers.

Enjoy your cultural exploration and the beautiful sights of Kyoto!